In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 大显存 GPU 重跑：attempt02

选择 **A100（优先）或其他显存超过16 GiB的 GPU**，然后“全部运行”。无需上传文件。
首次运行会先读取旧累计费用，再删除 `MyDrive/Model-WM/latent-control-20261116-attempt01/`。
只保留累计费用与操作数量，不保留该次失败堆栈、图像或日志；其他历史实验目录不删除。
GPU不符合要求时，在清理、下载和模型启动前停止。删除由你运行此 notebook 时执行。

新输出：`MyDrive/Model-WM/latent-control-20261116-attempt02/`。
这次授权一个新的完整执行预算：准备30分钟、模型6小时（含60秒封存预留）、CPU12核小时、RAM32GiB、结果10GiB。
显存上限改为当前单卡实际容量；旧费用不退还。重复点击只读，不自动产生第三次运行。
旧 `attempt01/summary.json` 或已经写好的 `attempt02-carry.json` 必须在原路径，无需手工操作。

作品、码本、seed=2026111601、512px/float32、25步、B/C机制、136作品与所有操作上限保持冻结。
DEV/000在旧运行中已生成；CAL/TEST未进入。两个CAL阈值封存后才进入TEST。
本轮仍须满足原来的质量、拒识、配对增益及完整成本通过条件。
权重只存 `/content`；同一运行时尽量复用包和权重，更换运行时后重新检查环境与缓存。

本入口没有修复梯度图的显存占用，而是执行你授权的更大显卡重跑。
CPU入口检查不等于真实GPU验证；更大显卡也不保证数值通过或全部预算内完成。
[冻结方法与通过条件](https://github.com/RICHAAARC/Model-WM/blob/main/latent_control/PREREGISTRATION.md)


In [ ]:
#@title 清理旧输出并启动一次大显存 GPU 重跑（重复点击只读）
import time, resource
_started = time.monotonic()
_r = resource.getrusage(resource.RUSAGE_SELF)
_cpu_start = _r.ru_utime + _r.ru_stime
import base64, io, json, pathlib, sys, tempfile, zipfile
_package = pathlib.Path(tempfile.mkdtemp(prefix="latent-control-frozen-", dir="/content"))
_payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_payload))) as _zip:
    for _name in _zip.namelist():
        if _package not in (_package / _name).resolve().parents:
            raise RuntimeError("Unsafe embedded package path")
    _zip.extractall(_package)
for _name in list(sys.modules):
    if any(_name == p or _name.startswith(p + ".") for p in ["latent_control", "wam_method", "wam_qualification"]):
        del sys.modules[_name]
sys.path.insert(0, str(_package))
from latent_control.rerun import run_colab
result = run_colab(started=_started, cpu_start=_cpu_start)
print(json.dumps(result, ensure_ascii=False, indent=2))
